# PatchTST Colab Experiments

This notebook is designed for running your repo directly from Google Drive in Colab.

The main workflow in this notebook focuses on:
- supervised PatchTST/42 and PatchTST/64 runs on `weather`, `electricity`, and `traffic`
- supervised hierarchical patching comparisons on the same datasets
- self-supervised linear probing and fine-tuning experiments
- transfer learning from `electricity` to `weather` and `traffic`


## Notes

- The notebook assumes your repo is stored in Google Drive.
- If you uploaded files manually, you do not need `git pull`.
- Your local `train.py` now supports `--padding-patch`, `--fc-dropout`, and `--head-dropout`.
- For the paper-style PatchTST variants, `padding_patch='end'` gives the expected extra trailing patch.

In [ ]:
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# Update this path if your Drive folder is different.
REPO_ROOT = Path('/content/drive/MyDrive/DLFinalProject/timeseries-transformer-1')
CODE_ROOT = REPO_ROOT / 'code'
DATA_DIR = REPO_ROOT / 'data'
CHECKPOINT_DIR = REPO_ROOT / 'checkpoints' / 'paper_horizons'
RESULTS_DIR = REPO_ROOT / 'results' / 'paper_horizons_colab'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

%cd /content/drive/MyDrive/DLFinalProject/timeseries-transformer-1

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print('Environment ready')
print('Repo root:', REPO_ROOT)
print('Data dir:', DATA_DIR)

In [ ]:
# Optional: use only if this Drive folder is a git repo.
# !git pull origin main

In [ ]:
!pip install -r code/requirements.txt

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Main Experiment Workflow

Use the cells below as the primary workflow for the final experiment matrix:

- Supervised PatchTST/42 and PatchTST/64 on `weather`, `electricity`, and `traffic`
- Supervised hierarchical patching comparisons on the same datasets
- Self-supervised linear probing and two-step fine-tuning on `weather`, `electricity`, and `traffic`
- Transfer learning by pretraining on `electricity` and evaluating on `weather` and `traffic`

In [ ]:
EXTENDED_DATASETS = {
    'weather': {
        'data_path': DATA_DIR / 'weather.csv',
        'pred_lens': [96, 192, 336, 720],
    },
    'electricity': {
        'data_path': DATA_DIR / 'electricity.csv',
        'pred_lens': [96, 192, 336, 720],
    },
    'traffic': {
        'data_path': DATA_DIR / 'traffic.csv',
        'pred_lens': [96, 192, 336, 720],
    },
}

SUPERVISED_VARIANTS = {
    'patchtst42': {
        'seq_len': 336,
        'patch_len': 16,
        'stride': 8,
        'padding_patch': 'end',
        'hierarchical_patching': False,
        'hierarchical_levels': 2,
        'hierarchical_merge_factor': 2,
    },
    'patchtst64': {
        'seq_len': 512,
        'patch_len': 16,
        'stride': 8,
        'padding_patch': 'end',
        'hierarchical_patching': False,
        'hierarchical_levels': 2,
        'hierarchical_merge_factor': 2,
    },
    'hier_patchtst42': {
        'seq_len': 336,
        'patch_len': 16,
        'stride': 8,
        'padding_patch': 'end',
        'hierarchical_patching': True,
        'hierarchical_levels': 2,
        'hierarchical_merge_factor': 2,
    },
    'hier_patchtst64': {
        'seq_len': 512,
        'patch_len': 16,
        'stride': 8,
        'padding_patch': 'end',
        'hierarchical_patching': True,
        'hierarchical_levels': 2,
        'hierarchical_merge_factor': 2,
    },
}

COMMON_MODEL_CONFIG = {
    'd_model': 128,
    'n_heads': 4,
    'n_layers': 3,
    'd_ff': 256,
    'dropout': 0.1,
    'attn_dropout': 0.0,
    'fc_dropout': 0.1,
    'head_dropout': 0.0,
    'batch_size': 128,
}

SUPERVISED_TRAINING_CONFIG = {
    'epochs': 100,
    'lr': 1e-4,
    'scheduler': 'type3',
    'patience': 20,
}

SELFSUP_PRETRAIN_CONFIG = {
    'seq_len': 512,
    'patch_len': 12,
    'stride': 12,
    'padding_patch': 'none',
    'pretrain_epochs': 100,
    'mask_ratio': 0.4,
    'pretrain_lr': 1e-4,
}

LINEAR_PROBE_CONFIG = {
    'linear_probe_epochs': 20,
    'probe_lr': 1e-4,
}

FINETUNE_CONFIG = {
    'linear_probe_epochs': 10,
    'finetune_epochs': 20,
    'probe_lr': 1e-4,
    'finetune_lr': 1e-4,
}

EXTENDED_SINGLE_PRED_LEN = None
# EXTENDED_SINGLE_PRED_LEN = 96

print('Extended datasets:', list(EXTENDED_DATASETS.keys()))
print('Supervised variants:', list(SUPERVISED_VARIANTS.keys()))
print('Single horizon override:', EXTENDED_SINGLE_PRED_LEN)

In [ ]:
import importlib
import pandas as pd
import subprocess
from torch.utils.data import DataLoader

from data.window_dataset import build_datasets
import models.patchtst as patchtst_module

patchtst_module = importlib.reload(patchtst_module)
PatchTST = patchtst_module.PatchTST
PatchTSTConfig = patchtst_module.PatchTSTConfig


def evaluate_checkpoint(checkpoint_path, data_path, split='test', batch_size=32, device='cpu'):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    config = PatchTSTConfig(**checkpoint['config'])

    bundle = build_datasets(
        data_path=data_path,
        seq_len=config.seq_len,
        pred_len=config.pred_len,
        val_ratio=checkpoint['val_ratio'],
        test_ratio=checkpoint['test_ratio'],
        scale=checkpoint['scale'],
    )

    dataset = bundle.val if split == 'val' else bundle.test
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    model = PatchTST(config, in_channels=checkpoint['in_channels']).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    mae = torch.nn.L1Loss()
    mse = torch.nn.MSELoss()
    total_mae = 0.0
    total_mse = 0.0
    count = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            pred = model(x)
            total_mae += mae(pred, y).item()
            total_mse += mse(pred, y).item()
            count += 1

    return {
        'mae': total_mae / max(1, count),
        'mse': total_mse / max(1, count),
    }


def get_pred_lens(dataset_name):
    pred_lens = EXTENDED_DATASETS[dataset_name]['pred_lens']
    return [EXTENDED_SINGLE_PRED_LEN] if EXTENDED_SINGLE_PRED_LEN is not None else pred_lens


def build_model_args(config):
    args = [
        '--d-model', str(config['d_model']),
        '--n-heads', str(config['n_heads']),
        '--n-layers', str(config['n_layers']),
        '--d-ff', str(config['d_ff']),
        '--dropout', str(config['dropout']),
        '--attn-dropout', str(config['attn_dropout']),
        '--fc-dropout', str(config['fc_dropout']),
        '--head-dropout', str(config['head_dropout']),
        '--batch-size', str(config['batch_size']),
    ]
    return args


def run_train_command(command):
    print(' '.join(command))
    subprocess.run(command, check=True)


def run_supervised_variant(dataset_name, variant_name):
    dataset_cfg = EXTENDED_DATASETS[dataset_name]
    variant_cfg = SUPERVISED_VARIANTS[variant_name]
    rows = []
    for pred_len in get_pred_lens(dataset_name):
        ckpt = CHECKPOINT_DIR / f'{dataset_name}_{variant_name}_supervised_pred{pred_len}.pt'
        cmd = [
            'python', 'code/train.py',
            '--data', str(dataset_cfg['data_path']),
            '--seq-len', str(variant_cfg['seq_len']),
            '--pred-len', str(pred_len),
            '--patch-len', str(variant_cfg['patch_len']),
            '--stride', str(variant_cfg['stride']),
            '--padding-patch', variant_cfg['padding_patch'],
            '--epochs', str(SUPERVISED_TRAINING_CONFIG['epochs']),
            '--lr', str(SUPERVISED_TRAINING_CONFIG['lr']),
            '--scheduler', SUPERVISED_TRAINING_CONFIG['scheduler'],
            '--patience', str(SUPERVISED_TRAINING_CONFIG['patience']),
            '--device', device,
            '--checkpoint', str(ckpt),
        ] + build_model_args(COMMON_MODEL_CONFIG)
        if variant_cfg['hierarchical_patching']:
            cmd += [
                '--hierarchical-patching',
                '--hierarchical-levels', str(variant_cfg['hierarchical_levels']),
                '--hierarchical-merge-factor', str(variant_cfg['hierarchical_merge_factor']),
            ]
        run_train_command(cmd)
        metrics = evaluate_checkpoint(ckpt, dataset_cfg['data_path'], split='test', batch_size=COMMON_MODEL_CONFIG['batch_size'], device=device)
        rows.append({
            'experiment': 'supervised',
            'dataset': dataset_name,
            'variant': variant_name,
            'hierarchical_patching': variant_cfg['hierarchical_patching'],
            'pred_len': pred_len,
            'test_mae': metrics['mae'],
            'test_mse': metrics['mse'],
            'checkpoint': str(ckpt),
        })
    return pd.DataFrame(rows)


def ensure_selfsup_pretrain(source_dataset_name):
    dataset_cfg = EXTENDED_DATASETS[source_dataset_name]
    pretrain_ckpt = CHECKPOINT_DIR / f'{source_dataset_name}_selfsup_pretrain.pt'
    if pretrain_ckpt.exists():
        print('Reusing pretrained checkpoint:', pretrain_ckpt)
        return pretrain_ckpt

    cmd = [
        'python', 'code/train.py',
        '--data', str(dataset_cfg['data_path']),
        '--seq-len', str(SELFSUP_PRETRAIN_CONFIG['seq_len']),
        '--pred-len', '96',
        '--patch-len', str(SELFSUP_PRETRAIN_CONFIG['patch_len']),
        '--stride', str(SELFSUP_PRETRAIN_CONFIG['stride']),
        '--padding-patch', SELFSUP_PRETRAIN_CONFIG['padding_patch'],
        '--pretrain-only',
        '--pretrain-epochs', str(SELFSUP_PRETRAIN_CONFIG['pretrain_epochs']),
        '--pretrain-mask-ratio', str(SELFSUP_PRETRAIN_CONFIG['mask_ratio']),
        '--pretrain-lr', str(SELFSUP_PRETRAIN_CONFIG['pretrain_lr']),
        '--device', device,
        '--checkpoint', str(pretrain_ckpt),
    ] + build_model_args(COMMON_MODEL_CONFIG)
    run_train_command(cmd)
    return pretrain_ckpt


def run_selfsup_downstream(target_dataset_name, mode='linear_probe', source_dataset_name=None):
    source_dataset_name = source_dataset_name or target_dataset_name
    pretrain_ckpt = ensure_selfsup_pretrain(source_dataset_name)
    dataset_cfg = EXTENDED_DATASETS[target_dataset_name]
    rows = []
    source_tag = source_dataset_name if source_dataset_name == target_dataset_name else f'{source_dataset_name}_to_{target_dataset_name}'

    for pred_len in get_pred_lens(target_dataset_name):
        ckpt = CHECKPOINT_DIR / f'{source_tag}_{mode}_pred{pred_len}.pt'
        cmd = [
            'python', 'code/train.py',
            '--data', str(dataset_cfg['data_path']),
            '--seq-len', str(SELFSUP_PRETRAIN_CONFIG['seq_len']),
            '--pred-len', str(pred_len),
            '--patch-len', str(SELFSUP_PRETRAIN_CONFIG['patch_len']),
            '--stride', str(SELFSUP_PRETRAIN_CONFIG['stride']),
            '--padding-patch', SELFSUP_PRETRAIN_CONFIG['padding_patch'],
            '--pretrained-checkpoint', str(pretrain_ckpt),
            '--scheduler', 'none',
            '--disable-early-stopping',
            '--device', device,
            '--checkpoint', str(ckpt),
        ] + build_model_args(COMMON_MODEL_CONFIG)

        if mode == 'linear_probe':
            cmd += [
                '--linear-probe-epochs', str(LINEAR_PROBE_CONFIG['linear_probe_epochs']),
                '--probe-lr', str(LINEAR_PROBE_CONFIG['probe_lr']),
            ]
        elif mode == 'finetune':
            cmd += [
                '--linear-probe-epochs', str(FINETUNE_CONFIG['linear_probe_epochs']),
                '--probe-lr', str(FINETUNE_CONFIG['probe_lr']),
                '--finetune-epochs', str(FINETUNE_CONFIG['finetune_epochs']),
                '--finetune-lr', str(FINETUNE_CONFIG['finetune_lr']),
            ]
        else:
            raise ValueError(f'Unsupported mode: {mode}')

        run_train_command(cmd)
        metrics = evaluate_checkpoint(ckpt, dataset_cfg['data_path'], split='test', batch_size=COMMON_MODEL_CONFIG['batch_size'], device=device)
        rows.append({
            'experiment': mode,
            'source_dataset': source_dataset_name,
            'target_dataset': target_dataset_name,
            'pred_len': pred_len,
            'test_mae': metrics['mae'],
            'test_mse': metrics['mse'],
            'checkpoint': str(ckpt),
        })
    return pd.DataFrame(rows)


In [ ]:
supervised_frames = []
for dataset_name in ['weather', 'electricity', 'traffic']:
    for variant_name in ['patchtst42', 'patchtst64', 'hier_patchtst42', 'hier_patchtst64']:
        supervised_frames.append(run_supervised_variant(dataset_name, variant_name))

supervised_results_df = pd.concat(supervised_frames, ignore_index=True)
supervised_results_df

In [ ]:
selfsup_frames = []
for dataset_name in ['weather', 'electricity', 'traffic']:
    selfsup_frames.append(run_selfsup_downstream(dataset_name, mode='linear_probe'))
    selfsup_frames.append(run_selfsup_downstream(dataset_name, mode='finetune'))

selfsup_results_df = pd.concat(selfsup_frames, ignore_index=True)
selfsup_results_df

In [ ]:
transfer_frames = []
for target_dataset in ['weather', 'traffic']:
    transfer_frames.append(run_selfsup_downstream(target_dataset, mode='linear_probe', source_dataset_name='electricity'))
    transfer_frames.append(run_selfsup_downstream(target_dataset, mode='finetune', source_dataset_name='electricity'))

transfer_results_df = pd.concat(transfer_frames, ignore_index=True)
transfer_results_df

In [ ]:
supervised_summary_path = RESULTS_DIR / 'extended_supervised_summary.csv'
selfsup_summary_path = RESULTS_DIR / 'extended_selfsup_summary.csv'
transfer_summary_path = RESULTS_DIR / 'extended_transfer_summary.csv'
combined_summary_path = RESULTS_DIR / 'extended_all_results_summary.csv'

supervised_results_df.to_csv(supervised_summary_path, index=False)
selfsup_results_df.to_csv(selfsup_summary_path, index=False)
transfer_results_df.to_csv(transfer_summary_path, index=False)

combined_results_df = pd.concat(
    [supervised_results_df, selfsup_results_df, transfer_results_df],
    ignore_index=True,
    sort=False,
)
combined_results_df.to_csv(combined_summary_path, index=False)

print('Saved supervised summary to', supervised_summary_path)
print('Saved self-supervised summary to', selfsup_summary_path)
print('Saved transfer summary to', transfer_summary_path)
print('Saved combined summary to', combined_summary_path)

combined_results_df